# Homework Starter — Stage 04: Data Acquisition and Ingestion
- Name: Thuy Chau Tran
- Date: 08/17/2026

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [1]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/moctran/Desktop/NYU/BootcampIV/bootcamp_thuychau_tran/homework/homework4

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [MISS]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

1 needed file(s) missing. Put them at the paths above, relative to:
  /Users/moctran/Desktop/NYU/BootcampIV/bootcamp_thuychau_tran/homework/homework4
If that folder looks wrong, you are running the notebook from the wrong place.


In [9]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); 
ALPHA_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
print("Loaded ALPHAVANTAGE_API_KEY?", bool(ALPHA_KEY))

Loaded ALPHAVANTAGE_API_KEY? True


In [12]:
def safe_stamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")
safe_stamp
def safe_filename(prefix: str, meta: Dict[str, str]) -> str:
    mid = "_".join([f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()])
    return f"{prefix}_{mid}_{safe_stamp()}.csv"

In [13]:
def validate_df(df: pd.DataFrame, required_cols: List[str], dtypes_map: Dict[str, str]) -> Dict[str, str]:
    msgs = {}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        msgs['missing_cols'] = f"Missing columns: {missing}"
    for col, dtype in dtypes_map.items():
        if col in df.columns:
            try:
                if dtype == 'datetime64[ns]':
                    pd.to_datetime(df[col])
                elif dtype == 'float':
                    pd.to_numeric(df[col])
            except Exception as e:
                msgs[f'dtype_{col}'] = f"Failed to coerce {col} to {dtype}: {e}"
    na_counts = df.isna().sum().sum()
    msgs['na_total'] = f"Total NA values: {na_counts}"
    return msgs

## Helpers (use or modify)

In [10]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [14]:
SYMBOL = "AAPL"

use_alpha = bool(ALPHA_KEY)
print("Using Alpha Vantage:", use_alpha)

if use_alpha:
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": SYMBOL,
        "outputsize": "compact",
        "apikey": ALPHA_KEY,
        "datatype": "json"
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    key = [k for k in js.keys() if "Time Series" in k]
    if not key:
        # Alpha Vantage replies HTTP 200 with a prose blob - not an error status - when
        # the free tier's daily cap is hit or the endpoint has moved to premium, so
        # raise_for_status() above sees nothing wrong. Say so and fall back.
        print("Alpha Vantage returned no series:", str(list(js.values())[0])[:150])
        use_alpha = False

if use_alpha:
    series = js[key[0]]
    df_api = (pd.DataFrame(series).T
              .rename_axis('date')
              .reset_index())
    # keep a couple columns and coerce types
    df_api = df_api[['date', '4. close']].rename(columns={'4. close': 'close'})
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])

if not use_alpha:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period="6mo", interval="1d", auto_adjust=False,
                          multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']

df_api = df_api.sort_values('date').reset_index(drop=True)
msgs = validate_df(df_api, required_cols=['date','close'], dtypes_map={'date':'datetime64[ns]','close':'float'})
print(msgs)

fname = safe_filename(prefix="api", meta={"source": "alpha" if use_alpha else "yfinance", "symbol": SYMBOL})
out_path = DATA_RAW / fname
df_api.to_csv(out_path, index=False)
print("Saved:", out_path)

Using Alpha Vantage: True
{'na_total': 'Total NA values: 0'}
Saved: data/raw/api_source-alpha_symbol-AAPL_20260817-082236.csv


In [ ]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [17]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average"
headers = {"User-Agent": "AFE-Course-Notebook/1.0 (contact: instructor@example.edu)"}
try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table', class_='wikitable')   # the components table
    rows = []
    for tr in table.find_all('tr'):
        cells = [td.get_text(strip=True) for td in tr.find_all(['td','th'])]
        if cells:
            rows.append(cells)
    # assume first row is header
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)
except Exception as e:
    print("Scrape failed (demoing with inline HTML).", e)
    html = """
    <table>
      <tr><th>Ticker</th><th>Price</th></tr>
      <tr><td>AAA</td><td>101.2</td></tr>
      <tr><td>BBB</td><td>98.7</td></tr>
    </table>
    """
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    for tr in soup.find_all('tr'):
        cells = [td.get_text(strip=True) for td in tr.find_all(['td','th'])]
        if cells:
            rows.append(cells)
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)

if 'Price' in df_scrape.columns:
    df_scrape['Price'] = pd.to_numeric(df_scrape['Price'], errors='coerce')

msgs2 = validate_df(df_scrape, required_cols=list(df_scrape.columns), dtypes_map={})
print(msgs2)

fname2 = safe_filename(prefix="scrape", meta={"site": "wikipedia", "table": "djia"})
out_path2 = DATA_RAW / fname2
df_scrape.to_csv(out_path2, index=False)
print("Saved:", out_path2)

{'na_total': 'Total NA values: 0'}
Saved: data/raw/scrape_site-wikipedia_table-djia_20260817-082337.csv


In [18]:
_ = save_csv(df_scrape, prefix='scrape', site='example', table='markets')

Saved data/raw/scrape_site-example_table-markets_20260817-082343.csv


## Documentation
- API Source: (https://www.alphavantage.co/query) (https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average)
- Scrape Source: (URL/table description)
- Assumptions & risks: The free tier allows **25 calls a day**. Past that the API still answers `200 OK` with an explanation instead of data - which is why the code checks for the series rather than trusting the status code (Alpha Vantage) Might be exposed to changes because people can change the structure of the data returned by the API. The validation function checks for required columns and data types, but if the API changes its response format, this code may break;
- Confirm `.env` is not committed.